# VAE Latent Space — Interactive UMAP Dashboard

Encodes all cells in a split, runs UMAP, then:
- Displays an interactive Plotly scatter **in this cell** (one trace per condition)
- Saves a standalone HTML dashboard to `OUT_DIR/umap_dashboard.html` that replicates the kymograph-viewer style: click a point → see membrane + nuclei side by side; box-select → grid of cells.

In [1]:
import sys, io, json, base64
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import umap
from PIL import Image
import plotly.graph_objects as go
from IPython.display import display, HTML

try:
    from tqdm.notebook import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

sys.path.insert(0, str(Path("..").resolve() / "src"))
from darth_vaeder.JS_models import LitVAE
from darth_vaeder.datamodules import MultinucDataModule

In [2]:
# ── config ────────────────────────────────────────────────────────────────────
CKPT      = "/mnt/efs/dl_jrc/student_data/S-JS/repos/Darth_VAEder/outputs/checkpoints/version_31/epoch=317-val_loss=1.5346.ckpt"
ZARR      = "/mnt/efs/dl_jrc/student_data/S-JS/multinucleation.zarr"
TABLE     = "/mnt/efs/dl_jrc/student_data/S-JS/repos/Darth_VAEder/outputs/cell_table.csv"
OUT_DIR   = Path("outputs/umap_explorer")
MAX_CELLS = None         # None = all cells; e.g. 5000 for faster iteration
THUMB_PX  = 80           # pixels per channel in thumbnail (side-by-side → 80×160 total)
img_size  = 96           # matches model: 256 for native, 96 for downsampled
EDGE_THRESHOLD = 5       # drop cropped cells (edge_run_px >= N); must match training run (v31 used 5)
# UMAP
N_NEIGHBORS = 15
MIN_DIST    = 0.1
SEED        = 42

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output  : {OUT_DIR.resolve()}")

Output  : /mnt/efs/dl_jrc/student_data/S-JS/repos/Darth_VAEder/Joaquin'scripts/outputs/umap_explorer


In [3]:
# ── model ─────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = LitVAE.load_from_checkpoint(CKPT, map_location=device)
model.eval()
print(f"z_dim={model.hparams.z_dim}  nc={model.hparams.nc}  device={device}")

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/efs/dl_jrc/student_data/S-JS/repos/Darth_VAEder/outputs/checkpoints/version_31/epoch=317-val_loss=1.5346.ckpt'

In [ ]:
# ── datamodule — encode ALL splits (train + val + test) ───────────────────────
# setup(None) builds all three datasets in one call and applies the edge filter
# once, so every cell encoded here passed the same threshold used during training.
from torch.utils.data import ConcatDataset, DataLoader

dm = MultinucDataModule(
    data_path=ZARR,
    cell_table_csv=TABLE,
    channels=(0, 1),
    batch_size=256,
    num_workers=4,
    augment=True,
    pin_memory=False,
    persistent_workers=False,
    img_size=img_size,
    edge_threshold=EDGE_THRESHOLD,
)
dm.setup(None)   # builds train + val + test datasets, edge filter applied once

# all_split_dataset = ConcatDataset([dm.train_dataset, dm.val_dataset, dm.test_dataset])
all_split_dataset = ConcatDataset([dm.test_dataset])

loader = DataLoader(all_split_dataset, batch_size=256, num_workers=4, shuffle=False)

# n_train = len(dm.train_dataset)
# n_val   = len(dm.val_dataset)
n_test  = len(dm.test_dataset)
print(f"Edge filter kept: train={n_train}  val={n_val}  test={n_test}  total={len(all_split_dataset)}")
print(f"Patch size: {img_size}×{img_size}")

  edge filter : dropped 1468/16678 cells (edge_run_px >= 5 px)
Edge filter kept: train=11540  val=2153  test=1517  total=1517
Patch size: 96×96


In [ ]:
# ── thumbnail helper ──────────────────────────────────────────────────────────
def make_thumbnail(img: np.ndarray, size: int = THUMB_PX) -> str:
    """(2, H, W) float32 normalised → base64 PNG, channels side by side."""
    panels = []
    for ch in range(img.shape[0]):
        arr = np.clip(img[ch], 0.0, 1.0)
        pil = Image.fromarray((arr * 255).astype(np.uint8), mode="L")
        pil = pil.resize((size, size), Image.LANCZOS)
        panels.append(np.array(pil))
    composite = np.concatenate(panels, axis=1)   # (size, 2*size)
    buf = io.BytesIO()
    Image.fromarray(composite, mode="L").save(buf, format="PNG", optimize=True)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

In [ ]:
# ── encode cells + generate thumbnails ───────────────────────────────────────
all_mu, all_idx, all_thumbs = [], [], []
all_condition, all_replicate, all_imgname, all_localidx = [], [], [], []

with torch.no_grad():
    for batch in tqdm(loader, desc="encoding"):
        x_img    = batch[model.hparams.image_key].to(device)
        mask     = batch[model.hparams.mask_key].to(device)
        nuc_mask = batch[model.hparams.nuc_mask_key].to(device)
        x_in     = torch.cat([x_img, mask.float(), nuc_mask.float()], dim=1)
        _, _, mu, _ = model.vae(x_in)

        all_mu.append(mu.cpu().numpy())
        all_idx.extend(batch["index"].tolist())

        imgs_np = batch[model.hparams.image_key].cpu().numpy()  # (B, 2, H, W)
        for i in range(len(imgs_np)):
            all_thumbs.append(make_thumbnail(imgs_np[i]))

        meta = batch["metadata"]
        all_condition.extend(meta["condition"])
        all_replicate.extend(meta["replicate"])
        all_imgname.extend(meta["image_name"])
        all_localidx.extend([int(v) for v in meta["local_cell_index"]])

mu_all = np.concatenate(all_mu, axis=0)   # (N, z_dim)
print(f"Encoded {len(mu_all)} cells  |  mu shape: {mu_all.shape}")

# optional subsampling
if MAX_CELLS and len(mu_all) > MAX_CELLS:
    rng  = np.random.default_rng(SEED)
    keep = rng.choice(len(mu_all), MAX_CELLS, replace=False)
    keep = np.sort(keep)
    mu_all        = mu_all[keep]
    all_idx       = [all_idx[i]       for i in keep]
    all_thumbs    = [all_thumbs[i]    for i in keep]
    all_condition = [all_condition[i] for i in keep]
    all_replicate = [all_replicate[i] for i in keep]
    all_imgname   = [all_imgname[i]   for i in keep]
    all_localidx  = [all_localidx[i]  for i in keep]
    print(f"Subsampled to {len(mu_all)} cells")

encoding:   0%|          | 0/6 [00:00<?, ?it/s]

Encoded 1517 cells  |  mu shape: (1517, 64)


In [ ]:
# ── save VAE embeddings (keyed by cell_idx) ───────────────────────────────────
EMB_DIR = Path("outputs/embeddings")
EMB_DIR.mkdir(parents=True, exist_ok=True)

_t = pd.read_csv(TABLE)
_key2ci = {
    (str(r.replicate), str(r.condition), str(r.image_name), int(r.local_cell_index)): int(r.cell_idx)
    for r in _t.itertuples()
}
cell_idx = np.array([
    _key2ci[(str(all_replicate[i]), str(all_condition[i]), str(all_imgname[i]), int(all_localidx[i]))]
    for i in range(len(mu_all))
], dtype=np.int64)

np.savez(EMB_DIR / "vae_v31.npz", Z=mu_all.astype(np.float32), cell_idx=cell_idx)
print(f"Saved {mu_all.shape} → {EMB_DIR / 'vae_v31.npz'}")

In [ ]:
# ── decode mu → reconstruction thumbnails ────────────────────────────────────
# Pass stored mu vectors directly through both decoders (no reparameterization
# noise) for deterministic reconstructions. Per-image min-max normalisation
# per channel because reconstructions are unbounded (no sigmoid in decoder).

RECON_BATCH = 256

def make_recon_thumbnail(imgs_2ch: np.ndarray, size: int = THUMB_PX) -> str:
    """(2, H, W) float32 reconstruction → base64 PNG, channels side by side."""
    panels = []
    for ch in range(2):
        arr = imgs_2ch[ch].astype(np.float32)
        lo, hi = arr.min(), arr.max()
        arr = (arr - lo) / (hi - lo + 1e-6)
        pil = Image.fromarray((arr * 255).astype(np.uint8), mode="L")
        pil = pil.resize((size, size), Image.LANCZOS)
        panels.append(np.array(pil))
    composite = np.concatenate(panels, axis=1)
    buf = io.BytesIO()
    Image.fromarray(composite, mode="L").save(buf, format="PNG", optimize=True)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

all_recon_thumbs = []
model.eval()
with torch.no_grad():
    for start in tqdm(range(0, len(mu_all), RECON_BATCH), desc="decoding"):
        z = torch.from_numpy(mu_all[start:start + RECON_BATCH]).to(device)
        x_cell = model.vae.decoderCell(z)   # (B, 2, H, W): ch0=membrane, ch1=pCellmask
        x_nuc  = model.vae.decoderNuc(z)    # (B, 2, H, W): ch0=nuclei,   ch1=pNucmask
        imgs_np = torch.stack([x_cell[:, 0], x_nuc[:, 0]], dim=1).cpu().numpy()
        for i in range(len(imgs_np)):
            all_recon_thumbs.append(make_recon_thumbnail(imgs_np[i]))

print(f"Decoded {len(all_recon_thumbs)} reconstruction thumbnails")

In [ ]:
# ── diagnostic: verify every encoded cell passed the edge filter ───────────────
_t = pd.read_csv(TABLE)
_key2run = {
    (str(r.replicate), str(r.condition), str(r.image_name), int(r.local_cell_index)): int(r.edge_run_px)
    for r in _t.itertuples()
}
plotted_runs = np.array([
    _key2run[(str(all_replicate[i]), str(all_condition[i]), str(all_imgname[i]), int(all_localidx[i]))]
    for i in range(len(all_condition))
])
failed = int((plotted_runs >= EDGE_THRESHOLD).sum())
print(f"Encoded {len(plotted_runs)} cells")
print(f"edge_run_px  min/median/max: {plotted_runs.min()}  {int(np.median(plotted_runs))}  {plotted_runs.max()}")
print(f"Cells with edge_run_px >= {EDGE_THRESHOLD} (MUST be 0): {failed}")
assert failed == 0, f"BUG: {failed} cropped cells leaked through the edge filter!"
print("✓ All encoded cells passed the edge threshold")

Encoded 1517 cells
edge_run_px  min/median/max: 0  0  3
Cells with edge_run_px >= 5 (MUST be 0): 0
✓ All encoded cells passed the edge threshold


In [ ]:
# ── UMAP ──────────────────────────────────────────────────────────────────────
reducer = umap.UMAP(
    n_neighbors=N_NEIGHBORS,
    min_dist=MIN_DIST,
    n_components=2,
    random_state=SEED,
    verbose=True,
)
xy = reducer.fit_transform(mu_all)   # (N, 2)
print(f"UMAP done: {xy.shape}")

/home/S-JS/conda/envs/darth-vaeder/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP(n_jobs=1, random_state=42, verbose=True)
Wed Jun 17 13:43:26 2026 Construct fuzzy simplicial set
Wed Jun 17 13:43:29 2026 Finding Nearest Neighbors
Wed Jun 17 13:43:35 2026 Finished Nearest Neighbor Search
Wed Jun 17 13:43:35 2026 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Wed Jun 17 13:43:38 2026 Finished embedding
UMAP done: (1517, 2)


In [ ]:
# ── UMAP input mosaic (green = membrane, blue = nuclei) ───────────────────────
import matplotlib.pyplot as plt

GRID_N  = 20
CELL_PX = 64

def _decode_thumb(b64: str, size: int = THUMB_PX):
    """base64 PNG (membrane|nuclei side-by-side) → two (size,size) uint8 arrays."""
    raw = base64.b64decode(b64.split(",", 1)[1])
    img = np.array(Image.open(io.BytesIO(raw)))
    return img[:, :size], img[:, size:]

def _autocontrast(arr: np.ndarray, lo_pct: float = 2, hi_pct: float = 98) -> np.ndarray:
    nz = arr[arr > 0]
    lo, hi = (np.percentile(nz, [lo_pct, hi_pct]) if len(nz) else (0, 255))
    arr = np.clip(arr.astype(np.float32), lo, hi)
    return ((arr - lo) / (hi - lo + 1e-6) * 255).astype(np.uint8)

# ── bin cells into grid ───────────────────────────────────────────────────────
x_min, x_max = xy[:, 0].min(), xy[:, 0].max()
y_min, y_max = xy[:, 1].min(), xy[:, 1].max()
x_edges = np.linspace(x_min, x_max, GRID_N + 1)
y_edges = np.linspace(y_min, y_max, GRID_N + 1)

xi = np.clip(np.digitize(xy[:, 0], x_edges) - 1, 0, GRID_N - 1)
yi = np.clip(np.digitize(xy[:, 1], y_edges) - 1, 0, GRID_N - 1)

bin_rep = {}
for i in range(len(xy)):
    bx, by = int(xi[i]), int(yi[i])
    cx = (x_edges[bx] + x_edges[bx + 1]) / 2
    cy = (y_edges[by] + y_edges[by + 1]) / 2
    dist = (xy[i, 0] - cx) ** 2 + (xy[i, 1] - cy) ** 2
    if (bx, by) not in bin_rep or dist < bin_rep[(bx, by)][0]:
        bin_rep[(bx, by)] = (dist, i)

# ── build RGB mosaic ──────────────────────────────────────────────────────────
input_mosaic_rgb = np.zeros((GRID_N * CELL_PX, GRID_N * CELL_PX, 3), dtype=np.uint8)

for (bx, by), (_, idx) in bin_rep.items():
    mem_ch, nuc_ch = _decode_thumb(all_thumbs[idx], size=THUMB_PX)
    mem_r = np.array(Image.fromarray(mem_ch).resize((CELL_PX, CELL_PX), Image.LANCZOS))
    nuc_r = np.array(Image.fromarray(nuc_ch).resize((CELL_PX, CELL_PX), Image.LANCZOS))
    mem_r = _autocontrast(mem_r)
    nuc_r = _autocontrast(nuc_r)
    row = (GRID_N - 1 - by) * CELL_PX
    col = bx * CELL_PX
    input_mosaic_rgb[row:row + CELL_PX, col:col + CELL_PX, 1] = mem_r  # G = membrane
    input_mosaic_rgb[row:row + CELL_PX, col:col + CELL_PX, 2] = nuc_r  # B = nuclei

fig, ax = plt.subplots(1, 1, figsize=(10, 10), facecolor="black")
ax.imshow(input_mosaic_rgb, aspect="equal", interpolation="nearest")
ax.set_title("UMAP grid — inputs  |  green: membrane  blue: nuclei",
             color="white", fontsize=13, pad=8)
ax.axis("off")
plt.tight_layout(pad=0.5)
out_mosaic = OUT_DIR / "umap_mosaic.png"
plt.savefig(out_mosaic, dpi=150, bbox_inches="tight", facecolor="black")
plt.show()
print(f"Saved : {out_mosaic}")
print(f"Grid  : {GRID_N}×{GRID_N}  |  {len(bin_rep)} / {GRID_N**2} bins occupied")

In [ ]:
# ── UMAP reconstruction mosaic ────────────────────────────────────────────────
# Same grid as the input mosaic but using decoded reconstructions.
# Green = membrane (ch0 of decoderCell), Blue = nuclei (ch0 of decoderNuc).
# Lets you compare what the model actually learned to reconstruct vs the input.

def _autocontrast(arr: np.ndarray, lo_pct: float = 2, hi_pct: float = 98) -> np.ndarray:
    nz = arr[arr > 0]
    lo, hi = (np.percentile(nz, [lo_pct, hi_pct]) if len(nz) else (0, 255))
    arr = np.clip(arr.astype(np.float32), lo, hi)
    return ((arr - lo) / (hi - lo + 1e-6) * 255).astype(np.uint8)

recon_mosaic_rgb = np.zeros((GRID_N * CELL_PX, GRID_N * CELL_PX, 3), dtype=np.uint8)

for (bx, by), (_, idx) in bin_rep.items():
    mem_ch, nuc_ch = _decode_thumb(all_recon_thumbs[idx], size=THUMB_PX)
    mem_r = np.array(Image.fromarray(mem_ch).resize((CELL_PX, CELL_PX), Image.LANCZOS))
    nuc_r = np.array(Image.fromarray(nuc_ch).resize((CELL_PX, CELL_PX), Image.LANCZOS))
    mem_r = _autocontrast(mem_r)
    nuc_r = _autocontrast(nuc_r)
    row = (GRID_N - 1 - by) * CELL_PX
    col = bx * CELL_PX
    recon_mosaic_rgb[row:row + CELL_PX, col:col + CELL_PX, 1] = mem_r  # G = membrane
    recon_mosaic_rgb[row:row + CELL_PX, col:col + CELL_PX, 2] = nuc_r  # B = nuclei

fig, ax = plt.subplots(1, 1, figsize=(10, 10), facecolor="black")
ax.imshow(recon_mosaic_rgb, aspect="equal", interpolation="nearest")
ax.set_title("UMAP grid — reconstructions  |  green: membrane  blue: nuclei",
             color="white", fontsize=13, pad=8)
ax.axis("off")
plt.tight_layout(pad=0.5)
out_recon_mosaic = OUT_DIR / "umap_recon_mosaic.png"
plt.savefig(out_recon_mosaic, dpi=150, bbox_inches="tight", facecolor="black")
plt.show()
print(f"Saved : {out_recon_mosaic}")

In [ ]:
# ── build shared data structure ───────────────────────────────────────────────
conditions = list(dict.fromkeys(all_condition))   # preserve order, unique

items = [
    {
        "condition": all_condition[i],
        "replicate": str(all_replicate[i]),
        "filename":  f"{all_replicate[i]}|{all_condition[i]}|{all_imgname[i]}|cell_{all_localidx[i]}",
        "thumbnail": all_thumbs[i],
    }
    for i in range(len(mu_all))
]

cond_to_global = {c: [] for c in conditions}
for i, c in enumerate(all_condition):
    cond_to_global[c].append(i)

classes = [
    {
        "name":    cond,
        "x":       xy[cond_to_global[cond], 0].tolist(),
        "y":       xy[cond_to_global[cond], 1].tolist(),
        "indices": cond_to_global[cond],
    }
    for cond in conditions
]

data_dict = {
    "z_dim":   int(model.hparams.z_dim),
    "split":   "all",
    "n_cells": len(items),
    "classes": classes,
    "items":   items,
}
print(f"Conditions: {conditions}")
print(f"Cells per condition: { {c: len(cond_to_global[c]) for c in conditions} }")

Conditions: ['MATURE', 'CTRL', 'CMs25d']
Cells per condition: {'MATURE': 564, 'CTRL': 306, 'CMs25d': 647}


In [ ]:
# ── interactive Plotly scatter (notebook cell) ─────────────────────────────────
PAL = [
    "#4e9de0", "#ff8c42", "#e040fb", "#d62728", "#2ca02c",
    "#9467bd", "#8c564b", "#e377c2", "#bcbd22", "#17becf",
]

fig = go.Figure()
for i, cls in enumerate(classes):
    fig.add_trace(go.Scatter(
        x=cls["x"],
        y=cls["y"],
        mode="markers",
        name=cls["name"],
        marker=dict(size=5, opacity=0.85, color=PAL[i % len(PAL)]),
        customdata=cls["indices"],
        hovertemplate=(
            f"<b>{cls['name']}</b><br>"
            "global_idx=%{customdata}<extra></extra>"
        ),
    ))

fig.update_layout(
    title=f"UMAP  (z_dim={model.hparams.z_dim}, all splits)",
    height=700,
    clickmode="event+select",
    dragmode="pan",
    margin=dict(l=50, r=20, t=60, b=50),
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(title="UMAP 1", gridcolor="#eee", zeroline=False),
    yaxis=dict(title="UMAP 2", gridcolor="#eee", zeroline=False),
    legend=dict(bgcolor="white", bordercolor="#ddd", borderwidth=1),
)
fig.show()

In [ ]:
# ── build + save standalone HTML dashboard ────────────────────────────────────
HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <title>VAE – Latent Space Explorer</title>
  <script src="https://cdn.plot.ly/plotly-2.30.0.min.js"></script>
  <style>
    *, *::before, *::after { box-sizing: border-box }
    body  { font-family: Arial, sans-serif; margin: 0; padding: 16px;
            background: #f0f0f0; color: #222 }
    h2    { margin: 0 0 4px }
    .sub  { color: #999; font-size: 13px; margin: 0 0 14px }
    #wrap      { display: flex; gap: 16px; align-items: flex-start }
    #plot-col  { flex: 3 }
    #panel-col { flex: 2; min-width: 420px; max-height: 760px;
                 overflow-y: auto; background: #fff; border-radius: 8px;
                 padding: 14px; box-shadow: 0 2px 8px #bbb }
    #panel-col h4 { margin: 0 0 10px; font-size: 15px }
    .hint { color: #ccc; font-style: italic }
    .ibox b    { font-size: 14px }
    .ibox .meta { font-size: 11px; color: #999; word-break: break-all; margin: 2px 0 0 }
    .img-bg { background: #0a0a0a; padding: 4px; border-radius: 4px;
               width: 100%; margin-top: 6px }
    .img-bg img { width: 100%; display: block; image-rendering: pixelated }
    .ch-labels { display: flex; width: 100%; border-bottom: 1px solid #eee; margin-top: 2px }
    .ch-labels span { flex: 1; text-align: center; font-size: 10px; color: #999; padding: 2px 0 }
    .ghdr { font-size: 13px; font-weight: bold; margin-bottom: 8px }
    .grid { display: grid; grid-template-columns: repeat(2, 1fr); gap: 6px }
    .tile { background: #111; border-radius: 3px; padding: 3px 3px 0 3px }
    .tile img  { width: 100%; display: block; image-rendering: pixelated }
    .tile .tlbl { font-size: 9px; color: #bbb; text-align: center; padding: 2px 0 3px }
  </style>
</head>
<body>
<h2>VAE Latent Space Explorer</h2>
<p class="sub">
  Click a point to view its cell image &nbsp;&middot;&nbsp;
  Box/lasso-select for a grid view &nbsp;&middot;&nbsp;
  Scroll to zoom &nbsp;&middot;&nbsp; Drag to pan<br>
  <small>Left panel: <b>membrane</b> &nbsp;|&nbsp; right panel: <b>nuclei</b>
  (both normalised to [0,1])</small>
</p>
<div id="wrap">
  <div id="plot-col"><div id="umap-plot"></div></div>
  <div id="panel-col">
    <h4>Cell Image Viewer</h4>
    <div id="panel-content">
      <p class="hint">&#x1F446; Click a point or box-select a region.</p>
    </div>
  </div>
</div>
<script>
const DATA = __DATA_JSON__;
const PAL = [
  "#4e9de0","#ff8c42","#e040fb","#d62728","#2ca02c",
  "#9467bd","#8c564b","#e377c2","#bcbd22","#17becf"
];
const traces = DATA.classes.map((cls, i) => ({
  x:          cls.x,
  y:          cls.y,
  mode:       "markers",
  type:       "scatter",
  name:       cls.name,
  customdata: cls.indices,
  marker:     { size: 5, opacity: 0.85, color: PAL[i % PAL.length] },
  hovertemplate: "<b>" + cls.name + "</b><br>idx=%{customdata}<extra></extra>",
}));
Plotly.newPlot("umap-plot", traces, {
  title:         { text: "UMAP  (z_dim=" + DATA.z_dim + ", " + DATA.split + " split)", font: { size: 16 } },
  height:        720,
  clickmode:     "event+select",
  dragmode:      "pan",
  margin:        { l: 50, r: 20, t: 50, b: 50 },
  plot_bgcolor:  "#fff",
  paper_bgcolor: "#fff",
  xaxis:         { title: "UMAP 1", gridcolor: "#eee" },
  yaxis:         { title: "UMAP 2", gridcolor: "#eee" },
  legend:        { bgcolor: "#fff", bordercolor: "#ddd", borderwidth: 1 },
}, { scrollZoom: true, responsive: true });
const panel = document.getElementById("panel-content");
function singleView(globalIdx) {
  const it = DATA.items[globalIdx];
  panel.innerHTML =
    `<div class="ibox">
       <b>${it.condition} &mdash; ${it.replicate}</b>
       <div class="meta">${it.filename}</div>
     </div>
     <div class="img-bg">
       <img src="${it.thumbnail}" title="${it.condition} | ${it.replicate}">
     </div>
     <div class="ch-labels"><span>membrane</span><span>nuclei</span></div>`;
}
function gridView(pts) {
  const MAX = 12, n = pts.length;
  let h =
    `<div class="ghdr">${n} point${n !== 1 ? "s" : ""} selected` +
    `${n > MAX ? " — first " + MAX + " shown" : ""}</div>` +
    `<div class="grid">`;
  for (let i = 0; i < Math.min(n, MAX); i++) {
    const it = DATA.items[pts[i].customdata];
    h += `<div class="tile">
            <img src="${it.thumbnail}" title="${it.filename}">
            <div class="tlbl">${it.condition} | ${it.replicate}</div>
          </div>`;
  }
  panel.innerHTML = h + `</div>`;
}
const el = document.getElementById("umap-plot");
el.on("plotly_click",    ev => singleView(ev.points[0].customdata));
el.on("plotly_selected", ev => { if (ev && ev.points.length) gridView(ev.points); });
</script>
</body>
</html>"""

# embed data and write
data_json  = json.dumps(data_dict, separators=(",", ":"))
html_out   = HTML_TEMPLATE.replace("__DATA_JSON__", data_json)
out_path   = OUT_DIR / "umap_dashboard.html"
out_path.write_text(html_out, encoding="utf-8")

size_mb = out_path.stat().st_size / 1e6
print(f"Saved: {out_path}  ({size_mb:.1f} MB)")
print(f"Open in a browser — click any point to inspect its membrane + nuclei channels.")

Saved: outputs/umap_explorer/umap_dashboard.html  (2.6 MB)
Open in a browser — click any point to inspect its membrane + nuclei channels.


In [ ]:
# ── build + save reconstruction HTML dashboard ────────────────────────────────
# Same layout/template as the input dashboard but thumbnails show decoded
# reconstructions. Lets you compare input vs reconstruction side by side
# by opening both HTMLs in separate browser tabs.

recon_items = [
    {
        "condition": all_condition[i],
        "replicate": str(all_replicate[i]),
        "filename":  f"{all_replicate[i]}|{all_condition[i]}|{all_imgname[i]}|cell_{all_localidx[i]}",
        "thumbnail": all_recon_thumbs[i],
    }
    for i in range(len(mu_all))
]

recon_data_dict = {
    "z_dim":   int(model.hparams.z_dim),
    "split":   "all",
    "n_cells": len(recon_items),
    "classes": [
        {
            "name":    cond,
            "x":       xy[cond_to_global[cond], 0].tolist(),
            "y":       xy[cond_to_global[cond], 1].tolist(),
            "indices": cond_to_global[cond],
        }
        for cond in conditions
    ],
    "items": recon_items,
}

recon_data_json = json.dumps(recon_data_dict, separators=(",", ":"))
recon_html_out  = HTML_TEMPLATE.replace("__DATA_JSON__", recon_data_json)
out_recon_path  = OUT_DIR / "umap_recon_dashboard.html"
out_recon_path.write_text(recon_html_out, encoding="utf-8")

size_mb = out_recon_path.stat().st_size / 1e6
print(f"Saved: {out_recon_path}  ({size_mb:.1f} MB)")
print("Open alongside umap_dashboard.html to compare input vs reconstruction.")